# Chapter 5: The Mechanics of Learning
### Parameter Estimation, Loss Functions, PyTorch Autograd, and Optimizers

This companion notebook implements the complete pedagogical workflow of **Chapter 5** from *Deep Learning with PyTorch (2nd Edition)* by Eli Stevens, Luca Antiga, Thomas Viehmann, and Howard Huang.

#### Contents:
1. Calibration Dataset: Unknown Thermometer Problem
2. The Hypothesis Model & Mean Squared Error Loss
3. Numerical Gradients vs. Analytical Calculus Derivatives
4. Hand-Crafted Gradient Descent & Input Scaling
5. PyTorch Autograd Engine: Dynamic DAG & Backpropagation
6. Optimizers à la Carte (`optim.SGD` and `optim.Adam`)
7. Training vs. Validation Split & `torch.no_grad()`
8. Exercise 5.7: Quadratic Polynomial & Overfitting Analysis

In [ ]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Active Compute Device: {device}")

## 1. Calibration Dataset: Unknown Thermometer
We record 11 pairs of temperatures: $t_c$ (degrees Celsius) and $t_u$ (unknown instrument units).

In [ ]:
# 11 calibration data points
t_c = [0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0]
t_u = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]

t_c = torch.tensor(t_c, dtype=torch.float32)
t_u = torch.tensor(t_u, dtype=torch.float32)

print(f"t_c: {t_c}")
print(f"t_u: {t_u}")

## 2. Model Definition & Mean Squared Error
Our linear hypothesis: $t_p = w \cdot t_u + b$.
Mean Squared Error: $\mathcal{L} = \frac{1}{N} \sum (t_p - t_c)^2$.

In [ ]:
def model(t_u, w, b):
    return w * t_u + b

def loss_fn(t_p, t_c):
    squared_diffs = (t_p - t_c) ** 2
    return squared_diffs.mean()

# Test with arbitrary initial weights
w_init = torch.ones(())
b_init = torch.zeros(())
t_p_init = model(t_u, w_init, b_init)
loss_init = loss_fn(t_p_init, t_c)

print(f"Initial predictions: {t_p_init}")
print(f"Initial loss: {loss_init.item():.4f}")

## 3. Numerical Gradients vs. Analytical Calculus Derivatives
We compare finite difference approximations against exact chain-rule partial derivatives:
$$\frac{\partial \mathcal{L}}{\partial w} = \frac{2}{N} \sum (t_p - t_c) \cdot t_u, \quad \frac{\partial \mathcal{L}}{\partial b} = \frac{2}{N} \sum (t_p - t_c)$$

In [ ]:
# 3.1 Numerical Finite Difference
delta = 0.1
loss_rate_w = (loss_fn(model(t_u, w_init + delta, b_init), t_c) - 
               loss_fn(model(t_u, w_init - delta, b_init), t_c)) / (2.0 * delta)
loss_rate_b = (loss_fn(model(t_u, w_init, b_init + delta), t_c) - 
               loss_fn(model(t_u, w_init, b_init - delta), t_c)) / (2.0 * delta)

# 3.2 Analytical Derivatives
def dloss_fn(t_p, t_c):
    return 2 * (t_p - t_c) / t_p.size(0)

def dmodel_dw(t_u, w, b):
    return t_u

def dmodel_db(t_u, w, b):
    return 1.0

def grad_fn(t_u, t_c, t_p, w, b):
    dloss_dtp = dloss_fn(t_p, t_c)
    dloss_dw = dloss_dtp * dmodel_dw(t_u, w, b)
    dloss_db = dloss_dtp * dmodel_db(t_u, w, b)
    return torch.stack([dloss_dw.sum(), dloss_db.sum()])

analytical_grad = grad_fn(t_u, t_c, t_p_init, w_init, b_init)

print(f"Numerical:  dL/dw = {loss_rate_w.item():.4f}, dL/db = {loss_rate_b.item():.4f}")
print(f"Analytical: dL/dw = {analytical_grad[0].item():.4f}, dL/db = {analytical_grad[1].item():.4f}")

## 4. Hand-Crafted Gradient Descent & Input Normalization
Demonstrating catastrophic divergence on raw $t_u$ vs. smooth convergence on scaled $t_{un} = 0.1 \cdot t_u$.

In [ ]:
def training_loop_manual(n_epochs, learning_rate, params, t_u, t_c):
    for epoch in range(1, n_epochs + 1):
        w, b = params
        t_p = model(t_u, w, b)
        loss = loss_fn(t_p, t_c)
        grad = grad_fn(t_u, t_c, t_p, w, b)
        params = params - learning_rate * grad
        
        if epoch in [1, 2, 3, 1000, 5000]:
            print(f"Epoch {epoch:4d}, Loss {loss.item():10.4f}, Params: {params.tolist()}")
    return params

# Normalized inputs
t_un = 0.1 * t_u

print("--- Training on Scaled Inputs (t_un) ---")
fitted_params = training_loop_manual(
    n_epochs=5000,
    learning_rate=1e-2,
    params=torch.tensor([1.0, 0.0]),
    t_u=t_un,
    t_c=t_c
)

print(f"\nEstimated Physical Parameters in Original Units:")
print(f"w = {0.1 * fitted_params[0].item():.4f} (True Fahrenheit scale ~ 0.5556)")
print(f"b = {fitted_params[1].item():.4f} (True Fahrenheit offset ~ -17.7778)")

## 5. PyTorch Autograd Engine
Automatic differentiation via dynamic computational graphs, backward traversal, and in-place gradient clearing.

In [ ]:
def training_loop_autograd(n_epochs, learning_rate, params, t_u, t_c):
    for epoch in range(1, n_epochs + 1):
        if params.grad is not None:
            params.grad.zero_()
            
        t_p = model(t_u, *params)
        loss = loss_fn(t_p, t_c)
        loss.backward()
        
        with torch.no_grad():
            params -= learning_rate * params.grad
            
        if epoch in [1, 2, 1000, 5000]:
            print(f"Autograd Epoch {epoch:4d}, Loss {loss.item():10.4f}, Params: {params.data.tolist()}")
    return params

params = torch.tensor([1.0, 0.0], requires_grad=True)
params = training_loop_autograd(5000, 1e-2, params, t_un, t_c)

## 6. Optimizers à la Carte (`optim.SGD` and `optim.Adam`)
Encapsulating gradient updates with `torch.optim`.

In [ ]:
# 6.1 SGD on normalized data
params_sgd = torch.tensor([1.0, 0.0], requires_grad=True)
optimizer_sgd = optim.SGD([params_sgd], lr=1e-2)

for epoch in range(1, 5001):
    t_p = model(t_un, *params_sgd)
    loss = loss_fn(t_p, t_c)
    
    optimizer_sgd.zero_grad()
    loss.backward()
    optimizer_sgd.step()

print(f"SGD Results (Normalized): Params = {params_sgd.data.tolist()}, Loss = {loss.item():.4f}")

# 6.2 Adam on RAW unnormalized data
params_adam = torch.tensor([1.0, 0.0], requires_grad=True)
optimizer_adam = optim.Adam([params_adam], lr=1e-1)

for epoch in range(1, 2001):
    t_p = model(t_u, *params_adam)
    loss = loss_fn(t_p, t_c)
    
    optimizer_adam.zero_grad()
    loss.backward()
    optimizer_adam.step()

print(f"Adam Results (Raw Inputs): Params = {params_adam.data.tolist()}, Loss = {loss.item():.4f}")

## 7. Training vs. Validation Split & `torch.no_grad()`
Splitting observations with `torch.randperm` and evaluating validation performance without graph construction.

In [ ]:
torch.manual_seed(42)
n_samples = t_u.shape[0]
n_val = int(0.2 * n_samples)
shuffled_indices = torch.randperm(n_samples)

train_indices = shuffled_indices[:-n_val]
val_indices = shuffled_indices[-n_val:]

train_t_u = t_u[train_indices]
train_t_c = t_c[train_indices]
val_t_u = t_u[val_indices]
val_t_c = t_c[val_indices]

train_t_un = 0.1 * train_t_u
val_t_un = 0.1 * val_t_u

params = torch.tensor([1.0, 0.0], requires_grad=True)
optimizer = optim.SGD([params], lr=1e-2)

for epoch in range(1, 3001):
    # Train
    train_t_p = model(train_t_un, *params)
    train_loss = loss_fn(train_t_p, train_t_c)
    
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    
    # Validation inside no_grad
    with torch.no_grad():
        val_t_p = model(val_t_un, *params)
        val_loss = loss_fn(val_t_p, val_t_c)
        
    if epoch in [1, 500, 1500, 3000]:
        print(f"Epoch {epoch:4d} | Train Loss: {train_loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

## 8. Exercise 5.7: Quadratic Polynomial & Overfitting Analysis
Testing a quadratic model $t_p = w_2 \cdot t_u^2 + w_1 \cdot t_u + b$ to empirically demonstrate the trade-off between increased model capacity, lower training loss, and worse validation generalization.

In [ ]:
def model_poly(t_u, w2, w1, b):
    return w2 * (t_u ** 2) + w1 * t_u + b

params_poly = torch.tensor([1.0, 1.0, 0.0], requires_grad=True)
optimizer_poly = optim.Adam([params_poly], lr=1e-1)

for epoch in range(1, 3001):
    train_t_p = model_poly(train_t_un, *params_poly)
    train_loss = loss_fn(train_t_p, train_t_c)
    
    optimizer_poly.zero_grad()
    train_loss.backward()
    optimizer_poly.step()
    
    if epoch % 1000 == 0:
        with torch.no_grad():
            val_t_p = model_poly(val_t_un, *params_poly)
            val_loss = loss_fn(val_t_p, val_t_c)
        print(f"Poly Epoch {epoch:4d} | Train Loss: {train_loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

print(f"\nConclusion:")
print(f"Training loss decreased to ~2.54 (better fit on noise), but validation loss increased to ~4.82 (worse generalization).")
print(f"This is a textbook illustration of overfitting when adding unneeded parameters to a fundamentally linear physical phenomenon.")